%md
# Final Verdict between LSTM vs AutoML

| Approach | Selected on | Test accuracy | Test F1 (weighted) | Behaviour |
|---|---|:--:|:--:|---|
| **AutoML (LightGBM)** | val F1 (CV) | **0.5834** | **0.5067** | Leans "Up" (internal-split recall 0.95); **best of all models** |
| LSTM manual sweep (best = `baseline`) | test F1 (2-way, no val) | 0.4940 | 0.4926 | Balanced; weakest setup (selected on test, no val split) |
| LSTM baseline (50 epochs) | test F1 | 0.4932 | 0.4932 | Balanced |
| LSTM random search (`rand_07`) | val F1 | 0.5060 | **0.4994** | Balanced: best-tuned LSTM |
| LSTM Optuna (`optuna_12`) | val F1 | 0.5005 | 0.4923 | Balanced |

**Reading this honestly:**

1. **Every model sits at ~0.50–0.51 accuracy** — the efficient-market floor for next-day direction. The LSTM did **not** beat the tabular baseline on accuracy, so day-to-day ordering carries **no exploitable signal** at this horizon and feature set. That is a legitimate, evidence-based conclusion, not a failure.
2. **By weighted F1, the LSTMs (~0.49–0.50) beat AutoML (0.45)** — not because they are more skillful, but because they make *balanced* calls. AutoML rode the market's mild up-drift and predicted "Up" ~84% of the time, which inflates accuracy but tanks F1. Since a usable trading signal must call both directions, on the metric this project optimises for (**F1**) the tuned LSTM is the better-behaved model.
3. **Optuna ≈ random search ≈ baseline** for the LSTM — confirming there is no hidden hyperparameter config that unlocks skill.

**Champion choice:**

- **Champion = AutoML LightGBM** — it wins on both accuracy and F1, and as a single-row sklearn model it serves trivially (no scaler/window handling). Registered as `raghavan_trading_signals.ml.signal_model` with the `@champion` alias.
- The tuned LSTM stays a **challenger** (`raghavan_trading_signals.ml.lstm_signal_model`, currently the Optuna run). To serve it anyway, use the **`rand_07`** checkpoint (the best LSTM, held-out F1 0.4994) via an MLflow `pyfunc` wrapper bundling the `StandardScaler` + 30-day windowing.

Model names (For my reference):
- AutoML (LightGBM) -> raghavan_trading_signals.ml.signal_model (Version 1)
- LSTM manual sweep (best = `baseline`) -> Did not save as the other models were much better
- LSTM baseline (50 epochs) -> raghavan_trading_signals.ml.lstm_signal_model (Version 5)
- LSTM random search (`rand_07`) -> raghavan_trading_signals.ml.lstm_signal_model (Version 6)
- LSTM Optuna (`optuna_12`) -> raghavan_trading_signals.ml.lstm_signal_model (Version 7)

## Registering the Champion Model

In [0]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

CATALOG = "raghavan_trading_signals"
user = spark.sql("SELECT current_user()").collect()[0][0]

## Pull the best AutoML run

In [0]:
AUTOML_EXPERIMENT_ID = "185006975530845"   # AutoML experiment (notebook 06)

automl_runs = mlflow.search_runs(
    experiment_ids=[AUTOML_EXPERIMENT_ID],
    order_by=["metrics.test_f1 DESC"],
    max_results=1,
)
automl_run_id = automl_runs.iloc[0]["run_id"]
automl_f1 = automl_runs.iloc[0].get("metrics.test_f1", float("nan"))
automl_acc = automl_runs.iloc[0].get("metrics.test_accuracy", float("nan"))
print(f"AutoML  best run {automl_run_id} | test F1 {automl_f1:.4f} | test acc {automl_acc:.4f}")

In [0]:
lstm_exp = mlflow.get_experiment_by_name(f"/Users/{user}/raghavan-trading-signals-lstm-tuning") #LSTM experiment (notebook 07)
lstm_runs = mlflow.search_runs(
    experiment_ids=[lstm_exp.experiment_id],
    filter_string="attributes.run_name LIKE 'best_%_final' and metrics.test_f1 > 0",
    order_by=["metrics.test_f1 DESC"],
    max_results=1,
)
lstm_run_id = lstm_runs.iloc[0]["run_id"]
lstm_f1 = lstm_runs.iloc[0].get("metrics.test_f1", float("nan"))
lstm_acc = lstm_runs.iloc[0].get("metrics.test_accuracy", float("nan"))
print(f"LSTM    best run {lstm_run_id} | test F1 {lstm_f1:.4f} | test acc {lstm_acc:.4f}")

In [0]:
print("=== CHAMPION DECISION ===")
print(f"AutoML test F1: {automl_f1:.4f}  (expect ~0.45, acc ~0.5110 — collapses to 'Up')")
print(f"LSTM   test F1: {lstm_f1:.4f}  (expect ~0.4994 for rand_07, acc ~0.5060 — balanced)")
print("Default: serve AutoML (Path A) — single-row, runs standalone here, simplest downstream.")
print("Alternative: serve the LSTM via the pyfunc wrapper (Path B) for the better balanced F1.")

In [0]:
if automl_f1 >= lstm_f1:
    champion_run_id, champion_kind, champion_f1, champion_acc = automl_run_id, "automl", automl_f1, automl_acc
else:
    champion_run_id, champion_kind, champion_f1, champion_acc = lstm_run_id, "lstm", lstm_f1, lstm_acc

print("=== CHAMPION DECISION ===")
print(f"AutoML  best run {automl_run_id} | test F1 {automl_f1:.4f} | test acc {automl_acc:.4f}")
print(f"LSTM    best run {lstm_run_id} | test F1 {lstm_f1:.4f} | test acc {lstm_acc:.4f}")
print(f"CHAMPION -> {champion_kind.upper()} (run {champion_run_id}) | test F1 {champion_f1:.4f} | test acc {champion_acc:.4f}")

## Register the AutoML model as champion

In [0]:
MODEL_NAME = f"{CATALOG}.ml.signal_model"   # the production champion name used by GRIND 2/3

# champion_run_id came from the decision cell above (= the AutoML run for your numbers).
assert champion_kind == "automl", "Champion is the LSTM — use Path B to serve it instead."
registered = mlflow.register_model(model_uri=f"runs:/{champion_run_id}/model", name=MODEL_NAME)
print(f"Registered {MODEL_NAME} version {registered.version}")

client.set_registered_model_alias(
    name=MODEL_NAME, alias="champion", version=registered.version
)
print(f"Alias 'champion' -> version {registered.version}")

Why an alias instead of a stage? Unity Catalog Model Registry uses aliases (@champion,
@challenger), not the legacy Staging/Production stages. Everything downstream loads
models:/{MODEL_NAME}@champion, so when you register a better model later you just move the alias
— zero code or endpoint changes.

## Create the serving endpoint

In [0]:
import requests, json
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
ENDPOINT = "raghavan-trading-signal-predictor"

cfg = {
    "name": ENDPOINT,
    "config": {"served_entities": [{
        "entity_name": MODEL_NAME,
        "entity_version": str(registered.version),
        "workload_size": "Small",
        "scale_to_zero_enabled": True,
    }]},
}
r = requests.post(f"https://{workspace_url}/api/2.0/serving-endpoints",
                  headers={"Authorization": f"Bearer {token}"}, json=cfg)
print(r.status_code, r.text[:500])

## Wait until READY, then test

In [0]:
import time
while True:
    r = requests.get(f"https://{workspace_url}/api/2.0/serving-endpoints/{ENDPOINT}",
                     headers={"Authorization": f"Bearer {token}"})
    ready = r.json().get("state", {}).get("ready", "NOT_READY")
    print("ready:", ready)
    if ready == "READY":
        break
    time.sleep(30)

## Validation

In [0]:
from pyspark.sql.functions import col

ROW_IDX = 1

metadata_cols = ["symbol","trade_date","open","high","low","close","adj_close","volume",
                 "dividends","stock_splits","prev_close","bronze_ingested_at",
                 "bronze_source_file","next_day_return","next_day_direction"]

# Pull several recent rows (newest first), then select one of them.
rows = (spark.table(f"{CATALOG}.gold.daily_features")
        .filter(col("symbol") == "AAPL")
        .orderBy(col("trade_date").desc())
        .limit(10)            
        .toPandas())

row = rows.iloc[[1]]
features = row.drop(columns=[c for c in metadata_cols if c in row.columns])

payload = {"dataframe_records": features.to_dict(orient="records")}
r = requests.post(f"https://{workspace_url}/serving-endpoints/{ENDPOINT}/invocations",
                  headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
                  json=payload)
print("trade_date:", row["trade_date"].iloc[0])
print(r.status_code, r.json())
print("Actual next-day direction:", int(row["next_day_direction"].iloc[0]))